# Simulation ALS Data: TAM3C2 + 4D-OBC Analysis

Time-Adaptive M3C2 using the new `py4dgeo.tam3c2` module on a simulated dataset.

**Dataset:** Sand Dune TLS Scans
- **Location:** `C:\rsa\research_proj\blender_project\simulation_test2\output\simulation_test2_als`
- **Format:** XYZ files

**Workflow:**
1. Load epochs (timestamps parsed from filenames)
2. Sample corepoints from the reference epoch
3. Build a `TAM3C2` algorithm object and hand it to `SpatiotemporalAnalysis`
4. `analysis.add_epochs(*others)` triggers per-target time-adaptive M3C2
5. Run a custom 4D-OBC region-growing algorithm
6. Visualize aggregation diagnostics + extracted objects

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
from datetime import datetime, timedelta

import numpy as np
import matplotlib.pyplot as plt

import py4dgeo
from py4dgeo import (
    TAM3C2,
    Weighting,
    extract_reference_and_others,
    sample_corepoints,
)
from py4dgeo.segmentation import RegionGrowingSeed, temporal_averaging
from py4dgeo.data_loader import read_pc_epochs_and_assign_timestamps

## 1. Configuration

In [ ]:
data_path = r'C:\rsa\research_proj\blender_project\simulation_test2\output\simulation_test2_als_downsampled'
output_path = os.path.join(os.getcwd(), 'simulation2_als_downsampled_tam3c2.zip')

reference_timestamp = datetime(2020, 1, 6, 0, 0, 0)
max_epochs_after_reference = None  # Use all available epochs

# Corepoint sampling - adjust for sand dune scale
corepoint_voxel_size = 0.5

# TAM3C2 parameters - adjust for sand dune scale and dynamics
normal_radii = [0.5]         # Smaller radii for finer features
max_window_ratio = [0.3]
required_points = 18
cyl_radius = 0.5
max_distance = 10.0
registration_error = 0.01
sigma_ratio = 1.0
space_time_ratio = 1.0
weighting = Weighting.GAUSSIAN
keep_neighborhoods = False

# 4D-OBC parameters
obc_neighborhood_radius = 0.5
obc_min_segments = 10
obc_minperiod = 3
obc_height_threshold = 0.05
obc_thresholds = [0.5, 0.6, 0.7, 0.8, 0.9]
obc_smoothing_window = 3

## 2. Load epochs and filter time range

In [ ]:
epochs = read_pc_epochs_and_assign_timestamps(folder=data_path, start_time=datetime(2020, 1, 1), time_increment=timedelta(days=1))

In [ ]:
# Limit the time range used in the analysis
if max_epochs_after_reference is not None and epochs:
    sorted_eps = sorted(epochs, key=lambda e: e.timestamp)
    ref_idx = next((i for i, e in enumerate(sorted_eps) if e.timestamp == reference_timestamp), None)
    if ref_idx is None:
        raise ValueError(f"Reference {reference_timestamp} not in data")
    epochs = sorted_eps[:ref_idx + 1 + max_epochs_after_reference]
    print(f"Using {len(epochs)} epochs ({epochs[0].timestamp} -> {epochs[-1].timestamp})")

In [ ]:
reference_epoch, other_epochs = extract_reference_and_others(epochs, reference_timestamp)
print(f"Reference: {reference_epoch.timestamp}  ({len(reference_epoch.cloud):,} pts)")
print(f"Other epochs: {len(other_epochs)}")

corepoints = sample_corepoints(reference_epoch, method='voxel', voxel_size=corepoint_voxel_size)
print(f"Sampled {len(corepoints):,} corepoints  (voxel={corepoint_voxel_size} m)")

## 3. Build TAM3C2 and run the spatiotemporal analysis

In [ ]:
tam = TAM3C2(
    epochs_timeseries=epochs,
    max_window_ratio=max_window_ratio,
    normal_radii=normal_radii,
    required_points=required_points,
    weighting=weighting,
    sigma_ratio=sigma_ratio,
    space_time_ratio=space_time_ratio,
    keep_neighborhoods=keep_neighborhoods,
    corepoints=corepoints,
    cyl_radius=cyl_radius,
    max_distance=max_distance,
    registration_error=registration_error,
)

analysis = py4dgeo.SpatiotemporalAnalysis(output_path, force=True)
analysis.reference_epoch = reference_epoch
analysis.corepoints = corepoints
analysis.m3c2 = tam

analysis.add_epochs(*other_epochs)
print(f"distances shape: {analysis.distances.shape}")
print(f"uncertainties shape: {analysis.uncertainties.shape}")

In [ ]:
# Temporal smoothing for 4D-OBC
analysis.smoothed_distances = temporal_averaging(
    analysis.distances, smoothing_window=obc_smoothing_window
)
print(f"smoothed shape: {analysis.smoothed_distances.shape}")

## 4. Aggregation diagnostics

In [ ]:
diag = tam.diagnostics()
for k, v in diag.items():
    if isinstance(v, np.ndarray):
        print(f"  {k}: shape={v.shape}, dtype={v.dtype}")
    elif isinstance(v, list):
        print(f"  {k}: list of length {len(v)}")

# Save for later inspection
tam.save_diagnostics(os.path.join(os.getcwd(), 'sand_dune_tam3c2_diag.npz'))

## Multi-scale analysis

In [ ]:
# Multi-scale planarity map on the (normal_radii x max_window_ratio) grid.
#
# Left  : mean planarity over a corepoint subsample for every grid cell.
#         This is the quantity the scale-selection "contest" maximises.
# Right : how often each cell wins that contest across all corepoints,
#         read directly from `diag['scale_idx']` (the cached optimal combo).

radii  = list(tam.normal_radii)      if hasattr(tam.normal_radii,      '__iter__') else [tam.normal_radii]
ratios = list(tam.max_window_ratio)  if hasattr(tam.max_window_ratio,  '__iter__') else [tam.max_window_ratio]
combos = tam._scale_combinations     # ordered: for sr in radii: for wr in ratios

# --- 1. subsample corepoints for speed (full sweep on all CPs is too slow) ---
rng = np.random.default_rng(0)
n_sub = min(2000, len(corepoints))
sub_idx = rng.choice(len(corepoints), size=n_sub, replace=False)

# --- 2. planarity for every (combo, corepoint) -----------------------------
ref_idx_in_ts = tam._find_epoch_index(reference_epoch)
ref_time      = reference_epoch.timestamp.timestamp()
time_range    = float(tam._epoch_times.max() - tam._epoch_times.min()) or 1.0

planarity = np.full((len(combos), n_sub), np.nan)
for k, (sr, wr) in enumerate(combos):
    mw = time_range * wr
    for j, i in enumerate(sub_idx):
        pts, *_ = tam._aggregate_sphere(corepoints[i], ref_time, ref_idx_in_ts, sr, mw)
        if pts is None or len(pts) < 3:
            continue
        pl, _ = tam._planarity_and_normal(pts)
        planarity[k, j] = pl

mean_pl = np.nanmean(planarity, axis=1).reshape(len(radii), len(ratios))

# --- 3. winning-scale frequency from cached diagnostics --------------------
diag = tam.diagnostics()
scale_idx_all = np.asarray(diag['scale_idx']).astype(int)
counts = np.bincount(scale_idx_all, minlength=len(combos))
freq   = (counts / counts.sum()).reshape(len(radii), len(ratios))

# --- 4. two-panel heatmap ---------------------------------------------------
fig, axs = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, M, title, cmap, fmt in [
    (axs[0], mean_pl, 'Mean planarity over corepoints',       'viridis', '.3f'),
    (axs[1], freq,    'Selection frequency of winning scale', 'magma',   '.1%'),
]:
    im = ax.imshow(M, origin='lower', aspect='auto', cmap=cmap)
    ax.set_xticks(range(len(ratios)))
    ax.set_xticklabels([f'{r:g}' for r in ratios])
    ax.set_yticks(range(len(radii)))
    ax.set_yticklabels([f'{r:g}' for r in radii])
    ax.set_xlabel('max_window_ratio')
    ax.set_ylabel('normal_radii [m]')
    ax.set_title(title)
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            val = M[i, j]
            if not np.isnan(val):
                ax.text(j, i, format(val, fmt), ha='center', va='center',
                        color='white', fontsize=10)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

print(f"Best (planarity) combo: {combos[int(np.nanargmax(mean_pl))]}")
print(f"Most-picked combo     : {combos[int(np.argmax(counts))]} "
      f"({counts.max()}/{counts.sum()} = {counts.max()/counts.sum():.1%})")

In [ ]:
# Cylinder-stage sensitivity to (normal_radii, max_window_ratio) on a DYNAMIC scene.
#
# For each (sr, wr) cell we (on a corepoint subsample, for ONE target epoch):
#   1. compute a normal on the reference epoch via _aggregate_sphere(sr, wr=wr)
#   2. run _aggregate_cylinder around the reference time AND the target time,
#      using that normal and a cylinder window mw = |t_tgt - t_ref| * wr
#   3. record along-normal spread, M3C2-style mean distance, and effective
#      sample count
#
# Three panels:
#   - mean along-normal spread (ref + tgt combined): drift contamination
#   - mean |M3C2 distance|                          : how much change is recovered
#   - mean num_samples (ref + tgt)                  : aggregation volume

radii  = list(tam.normal_radii)     if hasattr(tam.normal_radii,     '__iter__') else [tam.normal_radii]
ratios = list(tam.max_window_ratio) if hasattr(tam.max_window_ratio, '__iter__') else [tam.max_window_ratio]
combos = tam._scale_combinations  # ordered: for sr in radii: for wr in ratios

# --- 1. pick a target epoch that has accumulated the most change ----------
tgt_col = analysis.smoothed_distances.shape[1] - 1   # last target
tgt_epoch = other_epochs[tgt_col]
tgt_time  = tgt_epoch.timestamp.timestamp()
tgt_idx_in_ts = tam._find_epoch_index(tgt_epoch)

ref_idx_in_ts = tam._find_epoch_index(reference_epoch)
ref_time      = reference_epoch.timestamp.timestamp()
dt_abs        = abs(tgt_time - ref_time)
time_range    = float(tam._epoch_times.max() - tam._epoch_times.min()) or 1.0

# --- 2. corepoint subsample ------------------------------------------------
rng = np.random.default_rng(0)
n_sub = min(2000, len(corepoints))
sub_idx = rng.choice(len(corepoints), size=n_sub, replace=False)

# --- 3. per-cell statistics -----------------------------------------------
spread_map = np.full((len(combos), n_sub), np.nan)
dist_map   = np.full((len(combos), n_sub), np.nan)
nsamp_map  = np.full((len(combos), n_sub), np.nan)

for k, (sr, wr) in enumerate(combos):
    mw_sphere = time_range * wr        # used by sphere stage in tam3c2.py
    mw_cyl    = dt_abs    * wr         # used by cylinder stage in tam3c2.py
    for j, i in enumerate(sub_idx):
        cp = corepoints[i]

        # (a) normal at this combo
        pts_n, *_ = tam._aggregate_sphere(cp, ref_time, ref_idx_in_ts, sr, mw_sphere)
        if pts_n is None or len(pts_n) < 3:
            continue
        _, n = tam._planarity_and_normal(pts_n)

        # (b) cylinder around ref and tgt with this wr
        pts_r, *_ = tam._aggregate_cylinder(
            cp, n, ref_time, ref_idx_in_ts,
            tam.cyl_radius, tam.max_distance, mw_cyl,
        )
        pts_t, *_ = tam._aggregate_cylinder(
            cp, n, tgt_time, tgt_idx_in_ts,
            tam.cyl_radius, tam.max_distance, mw_cyl,
        )
        if pts_r is None or pts_t is None or len(pts_r) < 2 or len(pts_t) < 2:
            continue

        # along-normal projections (M3C2 convention)
        proj_r = (pts_r - cp) @ n
        proj_t = (pts_t - cp) @ n

        dist_map[k, j]   = proj_t.mean() - proj_r.mean()
        # combined spread: mean of the two within-neighbourhood stds
        spread_map[k, j] = 0.5 * (proj_r.std() + proj_t.std())
        nsamp_map[k, j]  = 0.5 * (len(proj_r) + len(proj_t))

mean_spread = np.nanmean(spread_map,         axis=1).reshape(len(radii), len(ratios))
mean_dist   = np.nanmean(np.abs(dist_map),   axis=1).reshape(len(radii), len(ratios))
mean_nsamp  = np.nanmean(nsamp_map,          axis=1).reshape(len(radii), len(ratios))

# --- 4. three-panel heatmap -----------------------------------------------
fig, axs = plt.subplots(1, 3, figsize=(17, 4.5))
panels = [
    (axs[0], mean_spread, 'Mean along-normal spread [m]\n(drift contamination)', 'magma',   '.3f'),
    (axs[1], mean_dist,   'Mean |M3C2 distance| [m]\n(recovered change)',         'viridis', '.3f'),
    (axs[2], mean_nsamp,  'Mean num_samples\n(ref+tgt cylinder, /2)',             'cividis', '.0f'),
]
for ax, M, title, cmap, fmt in panels:
    im = ax.imshow(M, origin='lower', aspect='auto', cmap=cmap)
    ax.set_xticks(range(len(ratios))); ax.set_xticklabels([f'{r:g}' for r in ratios])
    ax.set_yticks(range(len(radii)));  ax.set_yticklabels([f'{r:g}' for r in radii])
    ax.set_xlabel('max_window_ratio')
    ax.set_ylabel('normal_radii [m]')
    ax.set_title(title)
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            v = M[i, j]
            if not np.isnan(v):
                ax.text(j, i, format(v, fmt), ha='center', va='center',
                        color='white', fontsize=9)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

fig.suptitle(
    f'Cylinder-stage response on target {tgt_epoch.timestamp}  '
    f'(|Δt|={dt_abs:.0f}s)  on {n_sub} corepoints',
    y=1.02,
)
plt.tight_layout()
plt.show()


## 5. Data quality check

In [ ]:
sd = analysis.smoothed_distances
if sd is not None and sd.shape[1] > 0:
    valid_epochs_per_cp = np.sum(~np.isnan(sd), axis=1)
    max_abs_changes = np.nanmax(np.abs(sd), axis=1)

    has_enough = valid_epochs_per_cp >= obc_min_segments
    significant = max_abs_changes >= obc_height_threshold
    both = has_enough & significant

    print(f"Corepoints with >= {obc_min_segments} valid epochs: {has_enough.sum()} ({has_enough.mean()*100:.1f}%)")
    print(f"Corepoints with max |change| >= {obc_height_threshold} m: {significant.sum()} ({significant.mean()*100:.1f}%)")
    print(f"Meeting both: {both.sum()} ({both.mean()*100:.1f}%)")
    print(f"Max change: {np.nanmax(max_abs_changes):.4f} m   Mean change: {np.nanmean(max_abs_changes):.4f} m")
else:
    print("No distance data to analyze.")

## 7. Extract 4D-OBCs

In [ ]:

algo = py4dgeo.RegionGrowingAlgorithm(
    neighborhood_radius=obc_neighborhood_radius,
    min_segments=obc_min_segments,
    minperiod=obc_minperiod,
    height_threshold=obc_height_threshold,
    thresholds=obc_thresholds,
    seed_subsampling=1,
)

analysis.invalidate_results(seeds=True, objects=True, smoothed_distances=False)
objects = algo.run(analysis)
print(f"Extracted {len(objects)} 4D-OBCs from {len(analysis.seeds)} seeds")

## 8. Visualize results

In [ ]:
cp_idx_sel = 0

timestamps = [e.timestamp for e in other_epochs][:analysis.smoothed_distances.shape[1]]
ts = analysis.smoothed_distances[cp_idx_sel]

plt.figure(figsize=(12, 5))
plt.plot(timestamps, ts, c='black', ls='--', lw=0.7, label='time series')

for sid, s in enumerate(s for s in analysis.seeds if s.index == cp_idx_sel):
    plt.plot(
        timestamps[s.start_epoch:s.end_epoch + 1],
        ts[s.start_epoch:s.end_epoch + 1],
        lw=2, label=f'seed {sid}: {s.start_epoch}-{s.end_epoch} epochs'
    )
plt.xlabel('Time'); plt.ylabel('Distance [m]')
plt.title(f'Corepoint {cp_idx_sel}')
plt.xticks(rotation=45)
plt.grid(alpha=0.3); plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# plot the N-th object and its seed info
sel_object_idx = 0

if len(objects) > 0:
    sel_obj = analysis.objects[sel_object_idx]
    sel_seed = sel_obj.seed  # Use the seed stored on the object itself (NOT analysis.seeds[sel_object_idx])
    print(f"Object {sel_object_idx}: seed CP={sel_seed.index}, epochs {sel_seed.start_epoch}-{sel_seed.end_epoch}, size={len(sel_obj.indices)}")
    sel_obj.plot()

In [ ]:
# plot the N-th object and its seed info, with more details
from scipy.spatial import ConvexHull
from matplotlib.patches import Polygon
import matplotlib.colors as mcolors

sel_object_idx = 0 # object index to visualize

if len(objects) > 0:
    sel_object = analysis.objects[sel_object_idx]
    # get the seed from the object itself, NOT from analysis.seeds[i].
    sel_seed = sel_object.seed # seed index of the object
    seed_cp_idx = sel_seed.index # seed corepoint index

    fig, axs = plt.subplots(1, 2, figsize=(15, 5))
    ax1, ax2 = axs

    idxs = sel_object.indices
    epoch_of_interest = int(sel_object.end_epoch)
    magnitudes_of_interest = (analysis.smoothed_distances[:, epoch_of_interest] -
                              analysis.smoothed_distances[:, int(sel_object.start_epoch)])

    crange = 0.2
    cmap = plt.get_cmap('seismic_r').copy()
    norm = mcolors.CenteredNorm(halfrange=crange)
    cmapvals = norm(magnitudes_of_interest)

    for idx in idxs[::10]:
        ax1.plot(timestamps, analysis.smoothed_distances[idx],
                c=cmap(cmapvals[idx]), linewidth=0.5)
    ax1.plot(timestamps, analysis.smoothed_distances[seed_cp_idx],
            c='black', linewidth=1., label='Seed timeseries')
    ax1.axvspan(timestamps[sel_object.start_epoch], timestamps[sel_object.end_epoch],
               alpha=0.3, color='grey', label='4D-OBC timespan')
    ax1.legend()
    ax1.set_title('Time series of segmented 4D-OBC locations')
    ax1.set_xlabel('Date')
    ax1.set_ylabel('Distance [m]')
    ax1.grid(True, alpha=0.3)
    plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45)

    cloud = analysis.corepoints.cloud
    subset_cloud = cloud[idxs, :2]

    d = ax2.scatter(cloud[:, 0], cloud[:, 1], c=magnitudes_of_interest,
                   cmap='seismic_r', vmin=-crange, vmax=crange, s=1)
    plt.colorbar(d, format='%.2f', label='Change magnitude [m]', ax=ax2)

    if len(subset_cloud) >= 3:
        hull = ConvexHull(subset_cloud)
        ax2.add_patch(Polygon(subset_cloud[hull.vertices, 0:2],
                             label='4D-OBC hull', fill=False, edgecolor='black'))

    ax2.scatter(cloud[seed_cp_idx, 0], cloud[seed_cp_idx, 1],
               marker='*', s=200, c='black',
               label=f'Seed (CP {seed_cp_idx})', zorder=5)

    ax2.set_title('Spatial distribution of 4D-OBC')
    ax2.set_xlabel('X [m]')
    ax2.set_ylabel('Y [m]')
    ax2.legend(loc='upper right')
    ax2.axis('equal')
    plt.tight_layout()
    plt.show()
else:
    print("No objects to visualize!")